In [2]:
import random 
from collections import Counter
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
import os

llm = ChatOllama(
    model="llama3.2:latest",
    api_key=os.environ["OLLAMA_API_KEY"],
    base_url=os.environ["OLLAMA_BASE_URL"],
    )

system = SystemMessage(content="""
You are an AI agent playing a social deduction game.

Your goal depends on your role:
- If civilian: identify the spy
- If spy: avoid detection

Respond concisely and strategically.                      
""")

class player:
    def __init__(self, name):
        self.name = name
        self.role = None
        self.word = None
        self.clue = None
        self.suspicion = None
        self.agent = None
        self.personality = None
    

class players:
    def __init__(self):
        self.members = []
        self.alive = []
        self.votes = {}

class agents:
    def __init__(self, player):
        self.player = player

    def give_clue(self, players):
        self.player.clue = random.choice(clue_dict[self.player.word])
        return self.player.clue

    def vote(self, players):
        clues = {player: player.clue for player in players.alive}
        clue_counts = Counter(clues.values())
        suspicion = {}
        for other in players.alive:
            if other == self.player:
                continue
            clue = clues[other]
            suspicion[other] = 1 / clue_counts[clue]
        target = max(suspicion, key=suspicion.get)
        self.player.suspicion = target
        return target

class llmagent(agents):
    def give_clue(self, players):
        history = [player.clue for player in players.alive if player.clue]
        prompt = f"""
        You are playing a social deduction game.

        Your role: {self.player.role}
        Your word: {self.player.word}
        Your personality: {self.player.personality}
        {personality_str(self.player)}

        Give a ONE-WORD clue related to your word.
        Do not reveal the word directly.
        Do NOT repeat previous clues.
        Previous clues: {history}
        Avoid using the same word as others.
        """
        response = llm.invoke([system, HumanMessage(content=prompt)])
        self.player.clue = response.content.strip().split()[0]
        return self.player.clue

    def vote(self, players):
        clues_text = "\n".join([f"{player.name}: {player.clue}" for player in players.alive])
        prompt = f"""
        Players and their clues:

        {clues_text}

        One of them is the spy.

        Who is most suspicious?
        Reply ONLY with player name.
        """
        response = llm.invoke([system, HumanMessage(content=prompt)])
        name = response.content.strip()
        for player in players.alive:
            if player.name == name:
                return player
        return random.choice([player for player in players.alive if player.name != self.player.name])

num_players = 8
num_spies = 2

word_pairs = [
    ("Doctor", "Nurse"),
    ("Ocean", "River"),
    ("Laptop", "Tablet"),
    ("Plane", "Helicopter"),
    ("Pizza", "Burger"),
]

clue_dict = {
    "Apple": ["Red", "Fruit", "Sweet"],
    "Orange": ["Citrus", "Juicy", "Round"],
    "Cat": ["Pet", "Furry", "Meow"],
    "Dog": ["Bark", "Loyal", "Pet"],
}

def assign_role(players: players):
    """Assigns roles to players based on the number of players and spies."""
    spy_indices = random.sample(range(len(players.members)), num_spies)
    for i, player in enumerate(players.members):
        player.role = "spy" if i in spy_indices else "civilian"
    return players

def assign_words(players: players):
    """Assigns words to players based on their roles."""
    civilian_word, spy_word = random.choice(word_pairs)
    for player in players.members:
        if player.role == "spy":
            player.word = spy_word
        else:
            player.word = civilian_word
    return players

def clue(players: players):
    """Generates a clue based on the roles and words of the players."""
    players.votes = {}
    for player in players.alive:
        player.agent.give_clue(players)
    return players

def vote(players: players):
    """Votes on the clue of the players."""
    players.votes = {}
    for player in players.alive:
        target = player.agent.vote(players)
        players.votes[player] = target
    return players
 
def eliminate(players: players):
    """Eliminates players with most votes."""
    count = Counter(players.votes.values())
    if count:
        eliminated_player = count.most_common(1)[0][0]
        players.alive = [player for player in players.alive if player != eliminated_player]
    return players


def check_win(players: players):
    """Checks if the game has been won by either side."""
    spies_alive = [player for player in players.alive if player.role == "spy"]
    civilians_alive = [player for player in players.alive if player.role == "civilian"]
    
    if not spies_alive:
        return "Civilians win!"
    elif len(spies_alive) >= len(civilians_alive):
        return "Spies win!"
    else:
        return None
    
def act(players: players):
    """Performs one round of the game."""
    assign_role(players)
    assign_words(players)
    for player in players.alive:
        print(f"{player.name} has the word {player.word}.")
    round_count = 0
    while True:
        round_count += 1
        """Clue Phase"""
        for player in players.alive:
            player.agent.give_clue(players)
            print(f"{player.name} gives the clue: {player.clue}")
        """Vote Phase"""
        vote(players)
        for player, target in players.votes.items():
            print(f"{player.name} votes to eliminate {target.name}.")
        """Elimination & Check Win Phase"""
        eliminate(players)
        result = check_win(players)
        if result:
            return result, round_count
        print("Round over.")

def personality_str(player: player):
    """Returns a string representation of the personalities of the players."""
    if player.personality == "aggressive":
        return "You accuse quickly and take risks."
    elif player.personality == "conservative":
        return "You are cautious and think carefully before acting."
    elif player.personality == "deceptive":
        return "You are cunning and may not always tell the truth."
    else: 
        return "You play neutrally."

game = players()
game.members = [player(f"A{i}") for i in range(1, num_players+1)]
game.alive = game.members.copy()
game.votes = {}
for p in game.members:
    p.agent = llmagent(p)
    p.personality = random.choice(["aggressive", "conservative", "deceptive"])

# act(game)

In [3]:
def setup_game(names: list, personalities: list):
    """Creates and fully initialises a fresh game instance."""
    game = players()
    game.members = [player(name) for name in names]
    game.alive = game.members.copy()
    game.votes = {}
    for i, p in enumerate(game.members):
        p.personality = personalities[i % len(personalities)]
        p.agent = llmagent(p)
    assign_role(game)
    assign_words(game)
    return game
 
 
def run_simulation(n=50):
    """Runs n full game simulations and returns aggregated stats."""
 
    personalities = ["aggressive", "conservative", "deceptive"]
 
    # Result trackers
    results = {"spy": 0, "civilian": 0}
    rounds_data = []
 
    personality_total    = {p: 0 for p in personalities}
    personality_survival = {p: 0 for p in personalities}
    personality_spy_total = {p: 0 for p in personalities}
    personality_role_stats = {p: {"spy": 0, "civilian": 0} for p in personalities}
 
    for sim in range(n):
        print(f"\n{'═'*40}")
        print(f"  Simulation {sim + 1} / {n}")
        print(f"{'═'*40}")
 
        # Fresh game each iteration
        game = setup_game(
            names=[f"A{i}" for i in range(1, num_players + 1)],
            personalities=personalities
        )
 
        # Show starting words
        for p in game.members:
            print(f"  {p.name} ({p.role}, {p.personality}): word = {p.word}")
 
        # Run the game
        winner, rounds = act(game)
 
        # Record outcome
        result_label = "spy" if "Spies" in winner else "civilian"
        results[result_label] += 1
        rounds_data.append({"winner": result_label, "rounds": rounds})
 
        # Aggregate personality stats
        for p in game.members:
            personality_total[p.personality] += 1
            personality_role_stats[p.personality][p.role] += 1
            if p.role == "spy":
                personality_spy_total[p.personality] += 1
 
        for p in game.alive:
            personality_survival[p.personality] += 1
 
    # Derived rates
    personality_spy_rate      = {}
    personality_survival_rate = {}
 
    for p in personalities:
        total = personality_total[p]
        personality_spy_rate[p]      = personality_spy_total[p] / total if total > 0 else 0
        personality_survival_rate[p] = personality_survival[p]  / total if total > 0 else 0
 
    return {
        "results":                  results,
        "rounds_data":              rounds_data,
        "personality_total":        personality_total,
        "personality_role_stats":   personality_role_stats,
        "personality_spy_total":    personality_spy_total,
        "personality_spy_rate":     personality_spy_rate,
        "personality_survival":     personality_survival,
        "personality_survival_rate": personality_survival_rate,
    }

In [ ]:
stats = run_simulation(10)


════════════════════════════════════════
  Simulation 1 / 10
════════════════════════════════════════
  A1 (civilian, aggressive): word = Plane
  A2 (civilian, conservative): word = Plane
  A3 (spy, deceptive): word = Helicopter
  A4 (civilian, aggressive): word = Plane
  A5 (civilian, conservative): word = Plane
  A6 (civilian, deceptive): word = Plane
  A7 (civilian, aggressive): word = Plane
  A8 (spy, conservative): word = Helicopter
A1 has the word Helicopter.
A2 has the word Plane.
A3 has the word Plane.
A4 has the word Plane.
A5 has the word Plane.
A6 has the word Plane.
A7 has the word Helicopter.
A8 has the word Plane.
A1 gives the clue: Radar.
A2 gives the clue: Control
A3 gives the clue: Flight.
A4 gives the clue: Belt
A5 gives the clue: Gauge.
A6 gives the clue: Luggage
A7 gives the clue: Pilot.
A8 gives the clue: Engine
A1 votes to eliminate A8.
A2 votes to eliminate A4.
A3 votes to eliminate A8.
A4 votes to eliminate A6.
A5 votes to eliminate A2.
A6 votes to eliminate A2

In [ ]:
print("\n\n── Final Stats ──")
print(f"Results: {stats['results']}")
print(f"Personality survival rates: {stats['personality_survival_rate']}")
print(f"Personality spy rates:      {stats['personality_spy_rate']}")
print(f"Role stats: {stats['personality_role_stats']}")